In [ ]:
# Comparación de Modelos Predictivos
## Random Forest, Gradient Boosting, XGBoost, Prophet, LSTM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Cargar datos
df = pd.read_excel('../data/raw/dataset_sintetico_demanda_lima.xlsx')
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)
print(f"Datos cargados: {len(df)} días")
df.head()

In [ ]:
# Función para calcular métricas
def calcular_metricas(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

def plot_resultados(y_true, y_pred, title, model_name):
    plt.figure(figsize=(12, 5))
    plt.plot(y_true, label='Real', color='blue', alpha=0.7)
    plt.plot(y_pred, label=f'Predicho ({model_name})', color='red', alpha=0.7)
    plt.title(title)
    plt.xlabel('Tiempo (días)')
    plt.ylabel('Demanda')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Preparar features
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['dia'] = df['fecha'].dt.day

for lag in [1, 2, 3, 7]:
    df[f'lag_{lag}'] = df['demanda_real'].shift(lag)

df['media_movil_7'] = df['demanda_real'].rolling(7).mean()
df = df.dropna().reset_index(drop=True)

feature_cols = ['dia_semana', 'mes', 'dia', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'media_movil_7']
X = df[feature_cols].values
y = df['demanda_real'].values

# División train/test (80/20 respetando orden)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train: {len(X_train)} días")
print(f"Test: {len(X_test)} días")

In [ ]:
# 1. RANDOM FOREST
print("=" * 50)
print("Entrenando Random Forest...")
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
metrics_rf = calcular_metricas(y_test, y_pred_rf)
print(f"RF - MAE: {metrics_rf['MAE']:.2f}, RMSE: {metrics_rf['RMSE']:.2f}, MAPE: {metrics_rf['MAPE']:.1f}%")
plot_resultados(y_test, y_pred_rf, 'Random Forest', 'RF')

In [ ]:
# 2. GRADIENT BOOSTING
print("=" * 50)
print("Entrenando Gradient Boosting...")
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
metrics_gb = calcular_metricas(y_test, y_pred_gb)
print(f"GB - MAE: {metrics_gb['MAE']:.2f}, RMSE: {metrics_gb['RMSE']:.2f}, MAPE: {metrics_gb['MAPE']:.1f}%")
plot_resultados(y_test, y_pred_gb, 'Gradient Boosting', 'GB')

In [ ]:
# 3. XGBOOST
print("=" * 50)
print("Entrenando XGBoost...")
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
metrics_xgb = calcular_metricas(y_test, y_pred_xgb)
print(f"XGB - MAE: {metrics_xgb['MAE']:.2f}, RMSE: {metrics_xgb['RMSE']:.2f}, MAPE: {metrics_xgb['MAPE']:.1f}%")
plot_resultados(y_test, y_pred_xgb, 'XGBoost', 'XGB')

In [ ]:
# 4. PROPHET
print("=" * 50)
print("Entrenando Prophet...")
prophet_df = df[['fecha', 'demanda_real']].copy()
prophet_df.columns = ['ds', 'y']

train_prophet = prophet_df[:split_idx]
test_prophet = prophet_df[split_idx:]

model_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
model_prophet.fit(train_prophet)

future = model_prophet.make_future_dataframe(periods=len(test_prophet))
forecast = model_prophet.predict(future)

y_pred_prophet = forecast.iloc[-len(test_prophet):]['yhat'].values
metrics_prophet = calcular_metricas(test_prophet['y'].values, y_pred_prophet)
print(f"Prophet - MAE: {metrics_prophet['MAE']:.2f}, RMSE: {metrics_prophet['RMSE']:.2f}, MAPE: {metrics_prophet['MAPE']:.1f}%")
plot_resultados(test_prophet['y'].values, y_pred_prophet, 'Prophet', 'Prophet')

In [ ]:
# 5. LSTM (opcional - requiere TensorFlow)
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from tensorflow.keras.callbacks import EarlyStopping
    
    print("=" * 50)
    print("Entrenando LSTM...")
    
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df[['demanda_real']].values)
    
    window = 7
    X_lstm, y_lstm = [], []
    for i in range(window, len(scaled)):
        X_lstm.append(scaled[i-window:i, 0])
        y_lstm.append(scaled[i, 0])
    
    X_lstm = np.array(X_lstm).reshape(-1, window, 1)
    y_lstm = np.array(y_lstm)
    
    split_lstm = int(len(X_lstm) * 0.8)
    X_train_lstm, X_test_lstm = X_lstm[:split_lstm], X_lstm[split_lstm:]
    y_train_lstm, y_test_lstm = y_lstm[:split_lstm], y_lstm[split_lstm:]
    
    lstm_model = Sequential([
        LSTM(50, input_shape=(window, 1)),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')
    
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    lstm_model.fit(X_train_lstm, y_train_lstm, epochs=100, batch_size=16, 
                   validation_data=(X_test_lstm, y_test_lstm), callbacks=[early_stop], verbose=0)
    
    y_pred_lstm_scaled = lstm_model.predict(X_test_lstm)
    y_pred_lstm = scaler.inverse_transform(y_pred_lstm_scaled)
    y_test_lstm_original = scaler.inverse_transform(y_test_lstm.reshape(-1, 1))
    
    metrics_lstm = calcular_metricas(y_test_lstm_original.flatten(), y_pred_lstm.flatten())
    print(f"LSTM - MAE: {metrics_lstm['MAE']:.2f}, RMSE: {metrics_lstm['RMSE']:.2f}, MAPE: {metrics_lstm['MAPE']:.1f}%")
    plot_resultados(y_test_lstm_original.flatten(), y_pred_lstm.flatten(), 'LSTM', 'LSTM')
    
except ImportError:
    print(" TensorFlow no instalado. LSTM omitido.")
    metrics_lstm = None

In [ ]:
# COMPARACIÓN FINAL
print("\n" + "=" * 60)
print("RESUMEN DE COMPARACIÓN DE MODELOS")
print("=" * 60)

comparison = pd.DataFrame([
    {**metrics_rf, 'Modelo': 'Random Forest'},
    {**metrics_gb, 'Modelo': 'Gradient Boosting'},
    {**metrics_xgb, 'Modelo': 'XGBoost'},
    {**metrics_prophet, 'Modelo': 'Prophet'},
])

if metrics_lstm:
    comparison = pd.concat([comparison, pd.DataFrame([{**metrics_lstm, 'Modelo': 'LSTM'}])], ignore_index=True)

comparison = comparison.sort_values('MAE')
print(comparison.to_string(index=False))

# Gráfico comparativo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(['MAE', 'RMSE', 'MAPE']):
    sorted_df = comparison.sort_values(metric)
    axes[i].barh(sorted_df['Modelo'], sorted_df[metric], color='steelblue')
    axes[i].set_title(f'{metric} (menor es mejor)')
    axes[i].set_xlabel(metric)

plt.tight_layout()
plt.show()

# Mejor modelo
best_model = comparison.loc[comparison['MAE'].idxmin(), 'Modelo']
print(f"\n MEJOR MODELO: {best_model}")
print(f"   MAE: {comparison['MAE'].min():.2f}")
print(f"   MAPE: {comparison.loc[comparison['MAE'].idxmin(), 'MAPE']:.1f}%")